In [ ]:
# ==================== ШАГ 1: ИМПОРТЫ И НАСТРОЙКА ====================

import os
import json
import logging
import time
import warnings
from datetime import datetime
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from getpass import getpass
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# Suppress warnings
warnings.filterwarnings("ignore")

print("✅ Импорты загружены")

✅ Импорты загружены


In [ ]:
from google.colab import drive

drive.mount("/content/gdrive/")

print("✅ Google Drive успешно смонтирован!")

Drive already mounted at /content/gdrive/; to attempt to forcibly remount, call drive.mount("/content/gdrive/", force_remount=True).
✅ Google Drive успешно смонтирован!


In [ ]:
# ==================== ШАГ 2: КОНФИГУРАЦИЯ ====================

# Пути
OUTPUT_DIR = "/content/gdrive/My Drive/Colab Notebooks/ai24/diploma/"
DATA_FILE = "peft_102_questions_answers.csv"

# API конфигурация
API_URL = "https://api.vsellm.ru/v1/chat/completions"
API_KEY = getpass("API Key: ")

# STRONG_MODELS - генерация через vsellm API
STRONG_MODELS = [
    "google/gemini-3-flash-preview",
    "gemini-3-pro-preview",
    "gpt-5.2",
    "anthropic/claude-sonnet-4.5",
]

# HF_MODELS - генерация локально в Colab (с удалением после использования)
HF_MODELS = [
    "AvitoTech/avibe",
    "t-tech/T-lite-it-1.0",
    "RefalMachine/RuadaptQwen2.5-7B-Lite-Beta",
    "Qwen/Qwen2.5-7B-Instruct",
    "Qwen/Qwen2.5-Coder-7B-Instruct",
]

# JUDGE_MODELS - НЕ генерируют ответы, только оценивают через vsellm API
JUDGE_MODELS = ["qwen/qwen3-235b-a22b", "meta-llama/llama-3.3-70b-instruct"]

# Параметры генерации
TEMPERATURE = 0.7
MAX_TOKENS = 500
MAX_RETRIES = 3
RETRY_DELAY = 2
TIMEOUT = 60

# Промпт для генерации
AGILE_COACH_PROMPT = """### РОЛЬ
Ты опытный Agile коуч, который помогает командам в корпоративном мессенджере Mattermost.

### ЗАДАЧА
- Давать практичные, применимые советы по Agile методологиям
- Отвечать кратко и структурно (2-4 предложения или список из 2-3 пунктов)
- Использовать дружелюбный, но профессиональный тон
- Фокусироваться на конкретных действиях, а не теории
- Избегать длинных объяснений и академического стиля
- **Отвечать ТОЛЬКО на русском языке**

Вопрос сотрудника: {question}

Ответь на русском языке так, как ответил бы опытный коуч в быстрой переписке."""

print("✅ Конфигурация установлена")
print(f"STRONG_MODELS (vsellm): {len(STRONG_MODELS)}")
print(f"HF_MODELS (локально): {len(HF_MODELS)}")
print(f"JUDGE_MODELS (только оценка): {len(JUDGE_MODELS)}")

API Key: ··········
✅ Конфигурация установлена
STRONG_MODELS (vsellm): 4
HF_MODELS (локально): 5
JUDGE_MODELS (только оценка): 2


In [ ]:
# ==================== ШАГ 3: НАСТРОЙКА ЛОГИРОВАНИЯ ====================

# Создаем директории
output_path = Path(OUTPUT_DIR)
output_path.mkdir(parents=True, exist_ok=True)

log_dir = output_path / "logs"
log_dir.mkdir(parents=True, exist_ok=True)

# Настройка логгера
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
log_file = log_dir / f"evaluation_{timestamp}.log"

logger = logging.getLogger("LLM_Eval")
logger.setLevel(logging.DEBUG)

# Форматтер
formatter = logging.Formatter("%(asctime)s - %(levelname)s - %(message)s", datefmt="%H:%M:%S")

# File handler
fh = logging.FileHandler(log_file, encoding="utf-8")
fh.setLevel(logging.DEBUG)
fh.setFormatter(formatter)

# Console handler
ch = logging.StreamHandler()
ch.setLevel(logging.INFO)
ch.setFormatter(formatter)

logger.addHandler(fh)
logger.addHandler(ch)

logger.info("=" * 80)
logger.info("LLM EVALUATION STARTED")
logger.info("=" * 80)

print(f"✅ Логирование настроено: {log_file}")

15:03:27 - INFO - ================================================================================
15:03:27 - INFO - ================================================================================
15:03:27 - INFO - ================================================================================
INFO:LLM_Eval:================================================================================
15:03:27 - INFO - LLM EVALUATION STARTED
15:03:27 - INFO - LLM EVALUATION STARTED
15:03:27 - INFO - LLM EVALUATION STARTED
INFO:LLM_Eval:LLM EVALUATION STARTED
15:03:27 - INFO - ================================================================================
15:03:27 - INFO - ================================================================================
15:03:27 - INFO - ================================================================================
INFO:LLM_Eval:================================================================================


✅ Логирование настроено: /content/gdrive/My Drive/Colab Notebooks/ai24/diploma/logs/evaluation_20260129_150327.log


In [ ]:
# ==================== ШАГ 3.5: УСТАНОВКА БИБЛИОТЕК ДЛЯ HF ====================

!pip install -q transformers torch accelerate

print("✅ Библиотеки для HF установлены (без bitsandbytes)")

✅ Библиотеки для HF установлены (без bitsandbytes)


In [ ]:
# ==================== ШАГ 4: API CLIENTS (БЕЗ КВАНТИЗАЦИИ) ====================

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import gc


def create_session():
    """Создает HTTP session с retry логикой"""
    session = requests.Session()

    retry_strategy = Retry(
        total=MAX_RETRIES,
        backoff_factor=1,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["POST"],
    )

    adapter = HTTPAdapter(max_retries=retry_strategy)
    session.mount("http://", adapter)
    session.mount("https://", adapter)

    return session


def call_api_vsellm(model, messages, temperature=0.7, max_tokens=500):
    """Вызов vsellm API (для STRONG_MODELS и JUDGE_MODELS)"""
    headers = {"Content-Type": "application/json", "Authorization": f"Bearer {API_KEY}"}

    payload = {
        "model": model,
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens,
    }

    session = create_session()

    for attempt in range(MAX_RETRIES):
        try:
            response = session.post(API_URL, headers=headers, json=payload, timeout=TIMEOUT)

            response.raise_for_status()
            result = response.json()

            content = result.get("choices", [{}])[0].get("message", {}).get("content", "")

            if content:
                return content.strip()

        except Exception as e:
            logger.warning(f"VSELLM attempt {attempt + 1} failed for {model}: {str(e)}")
            time.sleep(RETRY_DELAY * (attempt + 1))

    logger.error(f"Failed to get response from {model}")
    return None


def generate_hf_local(model_name, question):
    """
    Генерация текста локально через HF модель (БЕЗ квантизации)
    Модель загружается, генерирует ответ, затем УДАЛЯЕТСЯ из памяти

    Args:
        model_name: Название модели
        question: Вопрос

    Returns:
        Сгенерированный текст или None
    """
    try:
        # Загружаем модель
        logger.info(f"Loading model {model_name}...")
        print(f"  🔄 Загрузка {model_name}...")

        tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

        # ✅ БЕЗ КВАНТИЗАЦИИ - просто FP16
        model = AutoModelForCausalLM.from_pretrained(
            model_name, device_map="auto", torch_dtype=torch.float16, trust_remote_code=True
        )

        print(f"  ✅ Модель загружена (FP16)")

        # Формируем промпт
        prompt = AGILE_COACH_PROMPT.format(question=question)
        full_prompt = f"System: Ты опытный Agile коуч.\n\nUser: {prompt}\n\nAssistant:"

        # Токенизация
        inputs = tokenizer(full_prompt, return_tensors="pt").to(model.device)

        # Генерация
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_TOKENS,
                temperature=TEMPERATURE,
                do_sample=True if TEMPERATURE > 0 else False,
                top_p=0.9,
                pad_token_id=tokenizer.eos_token_id,
            )

        # Декодирование
        generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

        # Извлекаем только ответ
        if "Assistant:" in generated_text:
            response = generated_text.split("Assistant:")[-1].strip()
        else:
            response = generated_text.strip()

        # ✅ ОЧИСТКА ПАМЯТИ
        del model
        del tokenizer
        del inputs
        del outputs
        torch.cuda.empty_cache()
        gc.collect()

        print(f"  ✅ Ответ получен, модель удалена из памяти")

        return response

    except Exception as e:
        logger.error(f"Error generating with {model_name}: {str(e)}")
        print(f"  ❌ Ошибка: {str(e)}")

        # Очистка памяти даже при ошибке
        try:
            del model
            del tokenizer
            torch.cuda.empty_cache()
            gc.collect()
        except:
            pass

        return None


print("✅ API clients готовы (vsellm + HF локально без квантизации)")

✅ API clients готовы (vsellm + HF локально без квантизации)


In [ ]:
# ==================== ШАГ 5: ЗАГРУЗКА ДАННЫХ ====================

data_path = output_path / DATA_FILE

logger.info(f"Loading data from: {data_path}")

df = pd.read_csv(data_path, sep=";")

df = df.replace(r"\n", " ", regex=True)

logger.info(f"Loaded {len(df)} rows")
logger.info(f"Columns: {list(df.columns)}")

print(f"✅ Данные загружены: {len(df)} строк")
print(f"Колонки: {list(df.columns)}")

# Показываем первые строки
display(df.head(3))

df = df[:2].copy()

15:03:32 - INFO - Loading data from: /content/gdrive/My Drive/Colab Notebooks/ai24/diploma/peft_102_questions_answers.csv
15:03:32 - INFO - Loading data from: /content/gdrive/My Drive/Colab Notebooks/ai24/diploma/peft_102_questions_answers.csv
15:03:32 - INFO - Loading data from: /content/gdrive/My Drive/Colab Notebooks/ai24/diploma/peft_102_questions_answers.csv
INFO:LLM_Eval:Loading data from: /content/gdrive/My Drive/Colab Notebooks/ai24/diploma/peft_102_questions_answers.csv
15:03:32 - INFO - Loaded 102 rows
15:03:32 - INFO - Loaded 102 rows
15:03:32 - INFO - Loaded 102 rows
INFO:LLM_Eval:Loaded 102 rows
15:03:32 - INFO - Columns: ['question', 'answer']
15:03:32 - INFO - Columns: ['question', 'answer']
15:03:32 - INFO - Columns: ['question', 'answer']
INFO:LLM_Eval:Columns: ['question', 'answer']


✅ Данные загружены: 102 строк
Колонки: ['question', 'answer']


,question,answer
0,"Привет. Подскажите, в каком случае можно/нужн...",Привет! Если договорились в команде и он выпол...
1,"Привет! Подскажите, пожалуйста, с кем можно п...",Скрам не запрещает и не разрешает работу с дед...
2,Цели помогают фокусироваться на результате спр...,Ты делаешь задачи исходя из целей. Приоритет -...


In [ ]:
df

,question,answer
0,"Привет. Подскажите, в каком случае можно/нужн...",Привет! Если договорились в команде и он выпол...
1,"Привет! Подскажите, пожалуйста, с кем можно п...",Скрам не запрещает и не разрешает работу с дед...


In [ ]:
# ==================== ШАГ 6: ГЕНЕРАЦИЯ ОТВЕТОВ ====================

logger.info("=" * 80)
logger.info("PHASE 1: GENERATING RESPONSES")
logger.info("=" * 80)

# ========== ЧАСТЬ 1: STRONG_MODELS через vsellm API ==========

logger.info("\n" + "=" * 80)
logger.info("GENERATING RESPONSES: STRONG MODELS (vsellm API)")
logger.info("=" * 80)

for model in STRONG_MODELS:
    column_name = f"{model.replace('/', '_')}_answer"

    if column_name in df.columns:
        logger.info(f"✓ {column_name} already exists, skipping")
        print(f"✓ {model} - уже есть")
        continue

    logger.info(f"Generating responses for: {model} (vsellm)")
    print(f"\n🔄 Генерация для {model} (vsellm API)...")

    answers = []
    errors = 0

    for idx, row in tqdm(df.iterrows(), total=len(df), desc=model):
        question = row["question"]
        prompt = AGILE_COACH_PROMPT.format(question=question)

        messages = [
            {"role": "system", "content": "Ты опытный Agile коуч."},
            {"role": "user", "content": prompt},
        ]

        answer = call_api_vsellm(
            model=model, messages=messages, temperature=TEMPERATURE, max_tokens=MAX_TOKENS
        )

        if answer:
            answers.append(answer)
        else:
            answers.append("[ERROR: No response]")
            errors += 1

        time.sleep(0.5)

    df[column_name] = answers

    # Сохраняем checkpoint
    checkpoint_path = output_path / f"checkpoint_after_{model.replace('/', '_')}.csv"
    df.to_csv(checkpoint_path, index=False)

    success_rate = ((len(answers) - errors) / len(answers) * 100) if len(answers) > 0 else 0
    logger.info(f"✓ {model}: {len(answers) - errors}/{len(answers)} ({success_rate:.1f}%)")
    print(f"✅ {model}: {len(answers) - errors}/{len(answers)} ({success_rate:.1f}%)")

print("\n✅ STRONG_MODELS завершены")

15:03:36 - INFO - ================================================================================
15:03:36 - INFO - ================================================================================
15:03:36 - INFO - ================================================================================
INFO:LLM_Eval:================================================================================
15:03:36 - INFO - PHASE 1: GENERATING RESPONSES
15:03:36 - INFO - PHASE 1: GENERATING RESPONSES
15:03:36 - INFO - PHASE 1: GENERATING RESPONSES
INFO:LLM_Eval:PHASE 1: GENERATING RESPONSES
15:03:36 - INFO - ================================================================================
15:03:36 - INFO - ================================================================================
15:03:36 - INFO - ================================================================================
INFO:LLM_Eval:================================================================================
15:03:36 - INFO - 
15:03:36 


🔄 Генерация для google/gemini-3-flash-preview (vsellm API)...


google/gemini-3-flash-preview:   0%|          | 0/2 [00:00<?, ?it/s]

15:03:55 - INFO - ✓ google/gemini-3-flash-preview: 2/2 (100.0%)
15:03:55 - INFO - ✓ google/gemini-3-flash-preview: 2/2 (100.0%)
15:03:55 - INFO - ✓ google/gemini-3-flash-preview: 2/2 (100.0%)
INFO:LLM_Eval:✓ google/gemini-3-flash-preview: 2/2 (100.0%)
15:03:55 - INFO - Generating responses for: gemini-3-pro-preview (vsellm)
15:03:55 - INFO - Generating responses for: gemini-3-pro-preview (vsellm)
15:03:55 - INFO - Generating responses for: gemini-3-pro-preview (vsellm)
INFO:LLM_Eval:Generating responses for: gemini-3-pro-preview (vsellm)


✅ google/gemini-3-flash-preview: 2/2 (100.0%)

🔄 Генерация для gemini-3-pro-preview (vsellm API)...


...

15:04:38 - INFO - ✓ gpt-5.2: 2/2 (100.0%)
15:04:38 - INFO - ✓ gpt-5.2: 2/2 (100.0%)
15:04:38 - INFO - ✓ gpt-5.2: 2/2 (100.0%)
INFO:LLM_Eval:✓ gpt-5.2: 2/2 (100.0%)
15:04:38 - INFO - Generating responses for: anthropic/claude-sonnet-4.5 (vsellm)
15:04:38 - INFO - Generating responses for: anthropic/claude-sonnet-4.5 (vsellm)
15:04:38 - INFO - Generating responses for: anthropic/claude-sonnet-4.5 (vsellm)
INFO:LLM_Eval:Generating responses for: anthropic/claude-sonnet-4.5 (vsellm)


✅ gpt-5.2: 2/2 (100.0%)

🔄 Генерация для anthropic/claude-sonnet-4.5 (vsellm API)...


anthropic/claude-sonnet-4.5:   0%|          | 0/2 [00:00<?, ?it/s]

15:04:57 - INFO - ✓ anthropic/claude-sonnet-4.5: 2/2 (100.0%)
15:04:57 - INFO - ✓ anthropic/claude-sonnet-4.5: 2/2 (100.0%)
15:04:57 - INFO - ✓ anthropic/claude-sonnet-4.5: 2/2 (100.0%)
INFO:LLM_Eval:✓ anthropic/claude-sonnet-4.5: 2/2 (100.0%)


✅ anthropic/claude-sonnet-4.5: 2/2 (100.0%)

✅ STRONG_MODELS завершены


In [ ]:
# ========== ЧАСТЬ 2: HF_MODELS локально (с удалением) ==========

logger.info("\n" + "=" * 80)
logger.info("GENERATING RESPONSES: HF MODELS (local, with cleanup)")
logger.info("=" * 80)

for model in HF_MODELS:
    column_name = f"{model.replace('/', '_')}_answer"

    if column_name in df.columns:
        logger.info(f"✓ {column_name} already exists, skipping")
        print(f"✓ {model} - уже есть")
        continue

    logger.info(f"Generating responses for: {model} (local)")
    print(f"\n🔄 Генерация для {model} (локально)...")

    answers = []
    errors = 0

    # ⚠️ Генерируем ответы БЕЗ tqdm, так как модель загружается для каждого вопроса
    for idx, row in df.iterrows():
        question = row["question"]

        print(f"  Вопрос {idx + 1}/{len(df)}", end="\r")

        answer = generate_hf_local(model_name=model, question=question)

        if answer:
            answers.append(answer)
        else:
            answers.append("[ERROR: No response]")
            errors += 1

    df[column_name] = answers

    # Сохраняем checkpoint
    checkpoint_path = output_path / f"checkpoint_after_{model.replace('/', '_')}.csv"
    df.to_csv(checkpoint_path, index=False)

    success_rate = ((len(answers) - errors) / len(answers) * 100) if len(answers) > 0 else 0
    logger.info(f"✓ {model}: {len(answers) - errors}/{len(answers)} ({success_rate:.1f}%)")
    print(f"\n✅ {model}: {len(answers) - errors}/{len(answers)} ({success_rate:.1f}%)")

print("\n✅ HF_MODELS завершены")

15:11:24 - INFO - 
15:11:24 - INFO - 
15:11:24 - INFO - 
INFO:LLM_Eval:
15:11:24 - INFO - GENERATING RESPONSES: HF MODELS (local, with cleanup)
15:11:24 - INFO - GENERATING RESPONSES: HF MODELS (local, with cleanup)
15:11:24 - INFO - GENERATING RESPONSES: HF MODELS (local, with cleanup)
INFO:LLM_Eval:GENERATING RESPONSES: HF MODELS (local, with cleanup)
15:11:24 - INFO - ================================================================================
15:11:24 - INFO - ================================================================================
15:11:24 - INFO - ================================================================================
INFO:LLM_Eval:================================================================================
15:11:24 - INFO - Generating responses for: AvitoTech/avibe (local)
15:11:24 - INFO - Generating responses for: AvitoTech/avibe (local)
15:11:24 - INFO - Generating responses for: AvitoTech/avibe (local)
INFO:LLM_Eval:Generating responses for: AvitoTec


🔄 Генерация для AvitoTech/avibe (локально)...
  🔄 Загрузка AvitoTech/avibe...


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

...

model-00004-of-00004.safetensors:   0%|          | 0.00/953M [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/219 [00:00<?, ?B/s]

  ✅ Модель загружена (FP16)


In [ ]:
df = df[
    [
        "question",
        "answer",
        "google_gemini-3-flash-preview_answer",
        "gemini-3-pro-preview_answer",
        "gpt-5.2_answer",
        "anthropic_claude-sonnet-4.5_answer",
    ]
].copy()

In [ ]:
# ==================== ШАГ 7: УСТАНОВКА БИБЛИОТЕК ДЛЯ МЕТРИК ====================

# Раскомментируйте при первом запуске:
!pip install -q nltk rouge-score sentence-transformers
import nltk

nltk.download("punkt")

from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from sentence_transformers import SentenceTransformer

print("✅ Библиотеки для метрик загружены")

In [ ]:
# ==================== ШАГ 8: ЗАГРУЗКА EMBEDDING МОДЕЛИ ====================

EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"

logger.info(f"Loading embedding model: {EMBEDDING_MODEL}")
print(f"🔄 Загрузка embedding модели...")

embedding_model = SentenceTransformer(EMBEDDING_MODEL)

logger.info("Embedding model loaded")
print("✅ Embedding модель загружена")

In [ ]:
# ==================== ШАГ 9: ФУНКЦИИ ДЛЯ РАСЧЕТА МЕТРИК ====================


def calculate_bleu(reference, hypothesis):
    """Расчет BLEU score"""
    try:
        ref_tokens = reference.split()
        hyp_tokens = hypothesis.split()

        smoothing = SmoothingFunction().method1
        score = sentence_bleu([ref_tokens], hyp_tokens, smoothing_function=smoothing)
        return score
    except:
        return 0.0


def calculate_rouge(reference, hypothesis):
    """Расчет ROUGE scores"""
    try:
        scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
        scores = scorer.score(reference, hypothesis)

        return {
            "rouge1": scores["rouge1"].fmeasure,
            "rouge2": scores["rouge2"].fmeasure,
            "rougeL": scores["rougeL"].fmeasure,
        }
    except:
        return {"rouge1": 0.0, "rouge2": 0.0, "rougeL": 0.0}


def calculate_cosine_similarity(reference, hypothesis):
    """Расчет косинусной близости"""
    try:
        embeddings = embedding_model.encode([reference, hypothesis])

        cos_sim = np.dot(embeddings[0], embeddings[1]) / (
            np.linalg.norm(embeddings[0]) * np.linalg.norm(embeddings[1])
        )

        return float(cos_sim)
    except:
        return 0.0


print("✅ Функции метрик готовы")

In [ ]:
# ==================== ШАГ 10: РАСЧЕТ АВТОМАТИЧЕСКИХ МЕТРИК ====================

logger.info("=" * 80)
logger.info("PHASE 2: CALCULATING AUTOMATIC METRICS")
logger.info("=" * 80)

# Находим все колонки с ответами
answer_columns = [col for col in df.columns if col.endswith("_answer")]

print(f"\n🔄 Расчет метрик для {len(answer_columns)} моделей...")

for col in answer_columns:
    prefix = col.replace("_answer", "")

    # Проверяем, не рассчитаны ли уже метрики
    if f"{prefix}_bleu" in df.columns:
        logger.info(f"✓ Metrics for {col} already exist, skipping")
        print(f"✓ {prefix} - метрики уже есть")
        continue

    logger.info(f"Calculating metrics for {col}")
    print(f"\n🔄 {prefix}...")

    bleu_scores = []
    rouge1_scores = []
    rouge2_scores = []
    rougeL_scores = []
    cosine_scores = []

    for idx, row in tqdm(df.iterrows(), total=len(df), desc=f"Metrics {prefix}"):
        reference = str(row["answer"])
        hypothesis = str(row[col])

        # BLEU
        bleu = calculate_bleu(reference, hypothesis)
        bleu_scores.append(bleu)

        # ROUGE
        rouge = calculate_rouge(reference, hypothesis)
        rouge1_scores.append(rouge["rouge1"])
        rouge2_scores.append(rouge["rouge2"])
        rougeL_scores.append(rouge["rougeL"])

        # Cosine Similarity
        cosine = calculate_cosine_similarity(reference, hypothesis)
        cosine_scores.append(cosine)

    # Добавляем метрики в DataFrame
    df[f"{prefix}_bleu"] = bleu_scores
    df[f"{prefix}_rouge1"] = rouge1_scores
    df[f"{prefix}_rouge2"] = rouge2_scores
    df[f"{prefix}_rougeL"] = rougeL_scores
    df[f"{prefix}_cosine"] = cosine_scores

    # Логируем средние значения
    logger.info(f"Average metrics for {col}:")
    logger.info(f"  BLEU: {np.mean(bleu_scores):.4f}")
    logger.info(f"  ROUGE-1: {np.mean(rouge1_scores):.4f}")
    logger.info(f"  ROUGE-2: {np.mean(rouge2_scores):.4f}")
    logger.info(f"  ROUGE-L: {np.mean(rougeL_scores):.4f}")
    logger.info(f"  Cosine: {np.mean(cosine_scores):.4f}")

    print(f"  BLEU: {np.mean(bleu_scores):.4f}")
    print(f"  ROUGE-1: {np.mean(rouge1_scores):.4f}")
    print(f"  Cosine: {np.mean(cosine_scores):.4f}")

    # Сохраняем checkpoint
    checkpoint_path = output_path / f"checkpoint_metrics_{prefix}.csv"
    df.to_csv(checkpoint_path, index=False)

print("\n✅ ВСЕ МЕТРИКИ РАССЧИТАНЫ")